In [1]:
import pandas as pd
import numpy as np
import os

# Training datasets

### Players DF reduced without context games

In [2]:
year = 2024

players_reduced_df = pd.read_csv(f"procesed_data/season_player_stats_{year}_reduced/all_players.csv", encoding='utf-8')
players_historical_df = pd.read_csv(f"procesed_data/player_mean_stats_reduced.csv", encoding='utf-8')
final_dataset = pd.DataFrame()

"""
th_player1_component1_1, ..., th_player7_componentM_Z, th_bench_usg, ..., ta_player1_component1_1, ..., ta_player7_componentM_Z, ta_bench_usg, ..., th_player1_historic_component1_1, ..., 
th_player7_historic_componentM_Z, ..., ta_player1_historic_component1_1, ..., ta_player7_historic_componentM_Z, team_home_wins (1 o 0)
"""

pca_cols_current = [c for c in players_reduced_df.columns if c.startswith('PCA')]
pca_cols_historic = [c for c in players_historical_df.columns if c.startswith('PCA')]

for game_id, game_df in players_reduced_df.groupby('game_id'):
    row = {}
    row['game_id'] = game_id
    for team, team_df in game_df.groupby('team'):
        # Team home
        if team_df.iloc[0]['location'] == 'Home':
            prefix = "th"
            if team_df.iloc[0]["win"] == 1:
                team_home_wins = 1
            else:
                team_home_wins = 0
            row['team_home_wins'] = team_home_wins
        else:
            prefix = "ta"
        
        team_df = team_df.sort_values(by='mp', ascending=False)
        top5_mp = team_df.head(5)['mp'].sum()
        bench_mp = team_df['mp'].sum() - top5_mp
        row[f"{prefix}_bench_usg"] = bench_mp / top5_mp if top5_mp > 0 else 0
        # Get the 7 players with the most minutes played
        for player_idx in range(min(7, len(team_df))):  # Asegura máximo 7 jugadores
            player = team_df.iloc[player_idx]
            player_count = player_idx + 1
            for col in pca_cols_current:
                row[f"{prefix}_player{player_count}_{col}"] = player[col]
            
            # Stats históricos (con manejo de errores)
            player_name = player['player']
            hist_data = players_historical_df[players_historical_df['Player'] == player_name]
            
            for col in pca_cols_historic:
                if not hist_data.empty:
                    row[f"{prefix}_player{player_count}_historic_{col}"] = hist_data[col].values[0]
                else:
                    row[f"{prefix}_player{player_count}_historic_{col}"] = 0
    final_dataset = pd.concat([final_dataset, pd.DataFrame([row])], ignore_index=True)

if not os.path.exists(f"final_data/season_players_dataset_{year}_reduced"):
    os.makedirs(f"final_data/season_players_dataset_{year}_reduced")
# Guardar resultado
final_dataset.to_csv(f"final_data/season_players_dataset_{year}_reduced/season_players_dataset_{year}.csv", index=False, encoding='utf-8-sig')

### Players DF without context games

In [3]:
year = 2024

players_df = pd.read_csv(f"raw_data/season_player_stats_{year}/all_players.csv", encoding='utf-8')
players_historical_df = pd.read_csv(f"procesed_data/player_mean_stats.csv", encoding='utf-8')
final_dataset = pd.DataFrame()

"""
th_player1_component1_1, ..., th_player7_componentM_Z, th_bench_usg, ..., ta_player1_component1_1, ..., ta_player7_componentM_Z, ta_bench_usg, ..., th_player1_historic_component1_1, ..., 
th_player7_historic_componentM_Z, ..., ta_player1_historic_component1_1, ..., ta_player7_historic_componentM_Z, team_home_wins (1 o 0)
"""

stats_cols_current = [c for c in players_df.columns if not c in ["player","team","game_id","location","opponent","team_score","opponent_score","win","played","mp"]]
stats_cols_historic = [c for c in players_historical_df.columns if not c in ["Player"]]

for game_id, game_df in players_df.groupby('game_id'):
    row = {}
    row['game_id'] = game_id
    for team, team_df in game_df.groupby('team'):
        # Team home
        if team_df.iloc[0]['location'] == 'Home':
            prefix = "th"
            if team_df.iloc[0]["win"] == 1:
                team_home_wins = 1
            else:
                team_home_wins = 0
            row['team_home_wins'] = team_home_wins
        else:
            prefix = "ta"
        
        team_df = team_df.sort_values(by='mp', ascending=False)
        top5_mp = team_df.head(5)['mp'].sum()
        bench_mp = team_df['mp'].sum() - top5_mp
        row[f"{prefix}_bench_usg"] = bench_mp / top5_mp if top5_mp > 0 else 0
        # Get the 7 players with the most minutes played
        for player_idx in range(min(7, len(team_df))):  # Asegura máximo 7 jugadores
            player = team_df.iloc[player_idx]
            player_count = player_idx + 1
            for col in stats_cols_current:
                row[f"{prefix}_player{player_count}_{col}"] = player[col]
            
            # Stats históricos (con manejo de errores)
            player_name = player['player']
            hist_data = players_historical_df[players_historical_df['Player'] == player_name]
            
            for col in stats_cols_historic:
                if not hist_data.empty:
                    row[f"{prefix}_player{player_count}_historic_{col}"] = hist_data[col].values[0]
                else:
                    row[f"{prefix}_player{player_count}_historic_{col}"] = 0
    final_dataset = pd.concat([final_dataset, pd.DataFrame([row])], ignore_index=True)

if not os.path.exists(f"final_data/season_players_dataset_{year}"):
    os.makedirs(f"final_data/season_players_dataset_{year}")
# Guardar resultado
final_dataset.to_csv(f"final_data/season_players_dataset_{year}/season_players_dataset_{year}.csv", index=False, encoding='utf-8-sig')

### Teams DF without context games

In [4]:
year = 2024

teams_df = pd.read_csv(f"raw_data/season_team_stats_{year}/all_teams.csv", encoding='utf-8')
teams_historical_df = pd.read_csv(f"raw_data/all_teams_data.csv", encoding='utf-8')
teams_historical_df = teams_historical_df[teams_historical_df["Year"]>= 2020]
final_dataset = pd.DataFrame()

"""
team_home_stat1, ..., team_home_statN, team_home_historic_stat1_1, ..., team_home_historic_statM_Z, 
team_away_stat1, ..., team_away_statN, team_away_historic_stat1_1, ..., team_away_historic_statM_Z, team_home_wins (1 o 0)
"""

cols_current = [c for c in teams_df.columns if not c.startswith('opp')]
cols_historic = [c for c in teams_historical_df.columns if not c in ["Year", "Team"]]

# teams_df = teams_df.rename(columns={col: f"team_home_{col}" for col in cols_current})
# teams_df = teams_df.rename(columns={col: f"team_away_{col}" for col in cols_current})

for game_id, game_df in teams_df.groupby('game_id'):
    row = {}
    row['game_id'] = game_id
    for location, team_df in game_df.groupby('Location'):
        # Team home
        if location == 'Home':
            prefix = "team_home"
            if team_df.iloc[0]["Win"] == 1:
                team_home_wins = 1
            else:
                team_home_wins = 0
            row['team_home_wins'] = team_home_wins
        else:
            prefix = "team_away"
        
        for col in cols_current:
            if col == 'game_id' or col == 'Location' or col == 'Win' or col == "Opponent" or col == "team":
                continue
            row[f"{prefix}_{col}"] = team_df[col].values[0]
        
        team_historical_df = teams_historical_df[teams_historical_df['Team'] == team_df.iloc[0]['team']]
        for team_year in team_historical_df['Year'].unique():
            for col in cols_historic:
                row[f"{prefix}_{str(team_year)}_historic_{col}"] = team_historical_df[col].values[0]
        
    final_dataset = pd.concat([final_dataset, pd.DataFrame([row])], ignore_index=True)

if not os.path.exists(f"final_data/season_teams_dataset_{year}"):
    os.makedirs(f"final_data/season_teams_dataset_{year}")
# Guardar resultado
final_dataset.to_csv(f"final_data/season_teams_dataset_{year}/season_teams_dataset_{year}.csv", index=False, encoding='utf-8-sig')

### Teams DF reduced without context games

In [5]:
year = 2024

teams_df = pd.read_csv(f"procesed_data/season_team_stats_{year}_reduced/all_teams.csv", encoding='utf-8')
teams_historical_df = pd.read_csv(f"procesed_data/all_teams_data_reduced.csv", encoding='utf-8')
final_dataset = pd.DataFrame()

"""
team_home_component1, ..., team_home_componentN_K, team_home_historic_component1_1, ..., team_home_historic_componentM_Z, 
team_away_component1, ..., team_away_componentN_K, team_away_historic_component1_1, ..., team_away_historic_componentM_Z, team_home_wins (1 o 0)
"""

pca_cols_current = [c for c in teams_df.columns if not (c.startswith('PCA6') or c.startswith('PCA7') or c.startswith('PCA8'))] # Components of the opponent are excluded
pca_cols_historic = [c for c in teams_historical_df.columns if c.startswith('PCA')] + ["Champion"]

# teams_df = teams_df.rename(columns={col: f"team_home_{col}" for col in cols_current})
# teams_df = teams_df.rename(columns={col: f"team_away_{col}" for col in cols_current})

for game_id, game_df in teams_df.groupby('game_id'):
    row = {}
    row['game_id'] = game_id
    for location, team_df in game_df.groupby('Location'):
        # Team home
        if location == 'Home':
            prefix = "team_home"
            if team_df.iloc[0]["Win"] == 1:
                team_home_wins = 1
            else:
                team_home_wins = 0
            row['team_home_wins'] = team_home_wins
        else:
            prefix = "team_away"
        
        for col in pca_cols_current:
            if col == 'game_id' or col == 'Location' or col == 'Win' or col == "Opponent" or col == "team":
                continue
            row[f"{prefix}_{col}"] = team_df[col].values[0]
        
        team_historical_df = teams_historical_df[teams_historical_df['Team'] == team_df.iloc[0]['team']]
        for team_year in team_historical_df['Year'].unique():
            for col in pca_cols_historic:
                row[f"{prefix}_{str(team_year)}_historic_{col}"] = team_historical_df[col].values[0]
        
    final_dataset = pd.concat([final_dataset, pd.DataFrame([row])], ignore_index=True)

if not os.path.exists(f"final_data/season_teams_dataset_{year}_reduced"):
    os.makedirs(f"final_data/season_teams_dataset_{year}_reduced")
# Guardar resultado
final_dataset.to_csv(f"final_data/season_teams_dataset_{year}_reduced/season_teams_dataset_{year}.csv", index=False, encoding='utf-8-sig')

### Player DF reduced with context games

In [6]:
from collections import deque
teams = ["ATL", "BOS", "BRK", "CHI", "CHO", "CLE", "DAL", "DEN", "DET", "GSW", "HOU", "IND", "LAC", "LAL", "MEM", "MIA", "MIL", "MIN", "NOP", "NYK", "OKC", "ORL", "PHI", "PHO", "POR", "SAC", "SAS", "TOR", "UTA", "WAS"]
game_id_order = {}
for team in teams:
    games = {}
    context_games = deque(maxlen=4)
    for i in range(4):
        context_games.append(-1)
    team_games_df = pd.read_csv(f"raw_data/season_team_stats_{year}/{team}.csv", encoding='utf-8')
    for game_id, game_location in zip(team_games_df['game_id'], team_games_df['Location']):
        games[game_id] = [game_location, context_games.copy()]
        context_games.append(game_id)
    game_id_order[team] = games


In [25]:
cola = deque(maxlen=4)
cola

deque([], maxlen=4)

In [7]:
"""
th_player1_component1_1, ..., th_player7_componentM_Z, th_bench_usg, ..., ta_player1_component1_1, ..., ta_player7_componentM_Z, ta_bench_usg, ..., th_player1_historic_component1_1, ..., 
th_player7_historic_componentM_Z, ..., ta_player1_historic_component1_1, ..., ta_player7_historic_componentM_Z, team_home_wins (1 o 0)
context1_th_player1_component1_1, ..., context4_th_player7_componentM_Z, context1_ta_player1_component1_1, ..., context4_ta_player7_componentM_Z,

contextN = partidos anteriores de la misma temporada
"""
players_reduced_df = pd.read_csv(f"procesed_data/season_player_stats_{year}_reduced/all_players.csv", encoding='utf-8')
players_without_context_df = pd.read_csv(f"final_data/season_players_dataset_{year}_reduced/season_players_dataset_{year}.csv", encoding='utf-8')
pca_cols_current = [c for c in players_reduced_df.columns if c.startswith('PCA')]

# Establecer el contexto de los partidos
prefixes = ['th', 'ta']
new_columns = {}

for prefix in prefixes:
    for i in range(1, 5):
        for player_idx in range(1, 8):
            for col in pca_cols_current:
                col_name = f"{prefix}_context{i}_player{player_idx}_{col}"
                new_columns[col_name] = np.zeros(len(players_without_context_df), dtype=float)

# Crear un DataFrame con las nuevas columnas
new_cols_df = pd.DataFrame(new_columns)

# Concatenar con el original
players_without_context_df = pd.concat([players_without_context_df, new_cols_df], axis=1)


for idx in range(len(players_without_context_df)):
    row = players_without_context_df.iloc[idx]
    game_id = row['game_id']

    for team, games_dict in game_id_order.items():
        if game_id in games_dict:
            game_location, context_games_original = games_dict[game_id]
            context_games = context_games_original.copy() 

            prefix = 'th' if game_location == 'Home' else 'ta'

            
            for i in range(1, 5):
                if not context_games:
                    continue
                context_game_id = context_games.pop()
                if context_game_id == -1:
                    continue
                for player_idx in range(1, 8):
                    # Extraer jugadores del partido de contexto
                    players_context = players_reduced_df[
                        (players_reduced_df['game_id'] == context_game_id) &
                        (players_reduced_df['location'] == game_location)
                    ].sort_values(by='mp', ascending=False)
                    # print(players_context)
                    player_row = players_context.iloc[player_idx - 1]

                    for col in pca_cols_current:
                        col_name = f"{prefix}_context{i}_player{player_idx}_{col}"
                        # print(col_name)
                        players_without_context_df.at[idx, col_name] = player_row[col]

players_without_context_df.to_csv(f"final_data/season_players_dataset_{year}_reduced/season_players_dataset_with_context_{year}.csv", index=False, encoding='utf-8-sig')

### Player DF with context games

In [8]:
from collections import deque
teams = ["ATL", "BOS", "BRK", "CHI", "CHO", "CLE", "DAL", "DEN", "DET", "GSW", "HOU", "IND", "LAC", "LAL", "MEM", "MIA", "MIL", "MIN", "NOP", "NYK", "OKC", "ORL", "PHI", "PHO", "POR", "SAC", "SAS", "TOR", "UTA", "WAS"]
game_id_order = {}
for team in teams:
    games = {}
    context_games = deque(maxlen=4)
    for i in range(4):
        context_games.append(-1)
    team_games_df = pd.read_csv(f"raw_data/season_team_stats_{year}/{team}.csv", encoding='utf-8')
    for game_id, game_location in zip(team_games_df['game_id'], team_games_df['Location']):
        games[game_id] = [game_location, context_games.copy()]
        context_games.append(game_id)
    game_id_order[team] = games


In [9]:
"""
th_player1_stat1, ..., th_player7_statZ, th_bench_usg, ..., ta_player1_stat1, ..., ta_player7_statZ, ta_bench_usg, ..., th_player1_historic_stat1, ..., 
th_player7_historic_statZ, ..., ta_player1_historic_stat1, ..., ta_player7_historic_statZ, team_home_wins (1 o 0)
context1_th_player1_stat1, ..., context4_th_player7_statZ, context1_ta_player1_stat1, ..., context4_ta_player7_statZ,

contextN = partidos anteriores de la misma temporada
"""
players_df = pd.read_csv(f"raw_data/season_player_stats_{year}/all_players.csv", encoding='utf-8')
players_without_context_df = pd.read_csv(f"final_data/season_players_dataset_{year}/season_players_dataset_{year}.csv", encoding='utf-8')
cols_current = [c for c in players_df.columns if not c in ["player","team","game_id","location","opponent","team_score","opponent_score","win","played","mp"]]

# Establecer el contexto de los partidos
prefixes = ['th', 'ta']
new_columns = {}

for prefix in prefixes:
    for i in range(1, 5):
        for player_idx in range(1, 8):
            for col in cols_current:
                col_name = f"{prefix}_context{i}_player{player_idx}_{col}"
                new_columns[col_name] = np.zeros(len(players_without_context_df), dtype=float)

# Crear un DataFrame con las nuevas columnas
new_cols_df = pd.DataFrame(new_columns)

# Concatenar con el original
players_without_context_df = pd.concat([players_without_context_df, new_cols_df], axis=1)


for idx in range(len(players_without_context_df)):
    row = players_without_context_df.iloc[idx]
    game_id = row['game_id']

    for team, games_dict in game_id_order.items():
        if game_id in games_dict:
            game_location, context_games_original = games_dict[game_id]
            context_games = context_games_original.copy() 

            prefix = 'th' if game_location == 'Home' else 'ta'

            
            for i in range(1, 5):
                if not context_games:
                    continue
                context_game_id = context_games.pop()
                if context_game_id == -1:
                    continue
                for player_idx in range(1, 8):
                    # Extraer jugadores del partido de contexto
                    players_context = players_df[
                        (players_df['game_id'] == context_game_id) &
                        (players_df['location'] == game_location)
                    ].sort_values(by='mp', ascending=False)
                    # print(players_context)
                    player_row = players_context.iloc[player_idx - 1]

                    for col in cols_current:
                        col_name = f"{prefix}_context{i}_player{player_idx}_{col}"
                        # print(col_name)
                        players_without_context_df.at[idx, col_name] = player_row[col]

players_without_context_df.to_csv(f"final_data/season_players_dataset_{year}/season_players_dataset_with_context_{year}.csv", index=False, encoding='utf-8-sig')

### Teams DF with context games

In [10]:
game_id_order = {}
for team in teams:
    games = {}
    context_games = deque(maxlen=4)
    for i in range(4):
        context_games.append(-1)
    team_games_df = pd.read_csv(f"raw_data/season_team_stats_{year}/{team}.csv", encoding='utf-8')
    for game_id, game_location in zip(team_games_df['game_id'], team_games_df['Location']):
        games[game_id] = [game_location, context_games.copy()]
        context_games.append(game_id)
    game_id_order[team] = games

In [11]:
"""
team_home_stat1, ..., team_home_statN, team_home_historic_stat1, ..., team_home_historic_statZ, 
team_away_stat1, ..., team_away_statN, team_away_historic_stat1, ..., team_away_historic_statZ, team_home_wins (1 o 0)
context1_team_home_stat1, ..., context4_team_home_statN, context1_team_away_stat1, ..., context4_team_away_statN
"""

teams_df = pd.read_csv(f"raw_data/season_team_stats_{year}/all_teams.csv", encoding='utf-8')
teams_without_context_df = pd.read_csv(f"final_data/season_teams_dataset_{year}/season_teams_dataset_{year}.csv", encoding='utf-8')

cols_current = [c for c in teams_df.columns if not c.startswith('opp')]

# Establecer el contexto de los partidos
prefixes = ['team_home', 'team_away']
new_columns = {}

for prefix in prefixes:
    for i in range(1, 5):
        for col in cols_current:
            if col == 'game_id' or col == 'Location' or col == 'Win' or col == 'team' or col == "Opponent":
                continue
            col_name = f"{prefix}_context{i}_{col}"
            new_columns[col_name] = np.zeros(len(teams_without_context_df), dtype=float)

# Crear un DataFrame con las nuevas columnas
new_cols_df = pd.DataFrame(new_columns)

# Concatenar con el original
teams_without_context_df = pd.concat([teams_without_context_df, new_cols_df], axis=1)


for idx in range(len(teams_without_context_df)):
    row = teams_without_context_df.iloc[idx]
    game_id = row['game_id']

    for team, games_dict in game_id_order.items():
        if game_id in games_dict:
            game_location, context_games_original = games_dict[game_id]
            context_games = context_games_original.copy() 

            prefix = 'team_home' if game_location == 'Home' else 'team_away'

            
            for i in range(1, 5):
                if not context_games:
                    continue
                context_game_id = context_games.pop()
                if context_game_id == -1:
                    continue

                # Extraer jugadores del partido de contexto
                team_context = teams_df[(teams_df['game_id'] == context_game_id) & (teams_df['Location'] == game_location)]
                # print(players_context)
                team_row = team_context.iloc[0]

                for col in cols_current:
                    if col == 'game_id' or col == 'Location' or col == 'Win' or col == 'team' or col == "Opponent":
                        continue
                    col_name = f"{prefix}_context{i}_{col}"
                    # print(col_name)
                    teams_without_context_df.at[idx, col_name] = team_row[col]

teams_without_context_df.to_csv(f"final_data/season_teams_dataset_{year}/season_teams_dataset_with_context_{year}.csv", index=False, encoding='utf-8-sig')

### Teams DF reduced with context games

In [12]:
game_id_order = {}
for team in teams:
    games = {}
    context_games = deque(maxlen=4)
    for i in range(4):
        context_games.append(-1)
    team_games_df = pd.read_csv(f"raw_data/season_team_stats_{year}/{team}.csv", encoding='utf-8')
    for game_id, game_location in zip(team_games_df['game_id'], team_games_df['Location']):
        games[game_id] = [game_location, context_games.copy()]
        context_games.append(game_id)
    game_id_order[team] = games

In [13]:
"""
team_home_component1_1, ..., team_home_componentM_Z, team_home_historic_component1_1, ..., team_home_historic_componentM_Z, 
team_away_component1_1, ..., team_away_componentM_Z, team_away_historic_component1_1, ..., team_away_historic_componentM_Z, team_home_wins (1 o 0)
context1_team_home_component1_1, ..., context4_team_home_componentM_Z, context1_team_away_component1_1, ..., context4_team_away_componentM_Z
"""

teams_df = pd.read_csv(f"procesed_data/season_team_stats_{year}_reduced/all_teams.csv", encoding='utf-8')
teams_without_context_df = pd.read_csv(f"final_data/season_teams_dataset_{year}_reduced/season_teams_dataset_{year}.csv", encoding='utf-8')

pca_cols_current = [c for c in teams_df.columns if not (c.startswith('PCA6') or c.startswith('PCA7') or c.startswith('PCA8'))]

# Establecer el contexto de los partidos
prefixes = ['team_home', 'team_away']
new_columns = {}

for prefix in prefixes:
    for i in range(1, 5):
        for col in pca_cols_current:
            if col == 'game_id' or col == 'Location' or col == 'Win' or col == 'team' or col == "Opponent":
                continue
            col_name = f"{prefix}_context{i}_{col}"
            new_columns[col_name] = np.zeros(len(teams_without_context_df), dtype=float)

# Crear un DataFrame con las nuevas columnas
new_cols_df = pd.DataFrame(new_columns)

# Concatenar con el original
teams_without_context_df = pd.concat([teams_without_context_df, new_cols_df], axis=1)


for idx in range(len(teams_without_context_df)):
    row = teams_without_context_df.iloc[idx]
    game_id = row['game_id']

    for team, games_dict in game_id_order.items():
        if game_id in games_dict:
            game_location, context_games_original = games_dict[game_id]
            context_games = context_games_original.copy() 

            prefix = 'team_home' if game_location == 'Home' else 'team_away'

            
            for i in range(1, 5):
                if not context_games:
                    continue
                context_game_id = context_games.pop()
                if context_game_id == -1:
                    continue

                # Extraer jugadores del partido de contexto
                team_context = teams_df[(teams_df['game_id'] == context_game_id) & (teams_df['Location'] == game_location)]
                # print(players_context)
                team_row = team_context.iloc[0]

                for col in pca_cols_current:
                    if col == 'game_id' or col == 'Location' or col == 'Win' or col == 'team' or col == "Opponent":
                        continue
                    col_name = f"{prefix}_context{i}_{col}"
                    # print(col_name)
                    teams_without_context_df.at[idx, col_name] = team_row[col]

teams_without_context_df.to_csv(f"final_data/season_teams_dataset_{year}_reduced/season_teams_dataset_with_context_{year}.csv", index=False, encoding='utf-8-sig')

# Test datasets

In [14]:
teams = ["ATL", "BOS", "BRK", "CHI", "CHO", "CLE", "DAL", "DEN", "DET", "GSW", "HOU", "IND", "LAC", "LAL", "MEM", "MIA", "MIL", "MIN", "NOP", "NYK", "OKC", "ORL", "PHI", "PHO", "POR", "SAC", "SAS", "TOR", "UTA", "WAS"]
last_games_by_team = {}
for team in teams:
    df = pd.read_csv(f"raw_data/season_team_stats_{year}/{team}.csv", encoding='utf-8')
    
    # Ensure chronological order (assuming there's a 'date' column or game_id reflects order)
    df = df.sort_values(by="game_id")  # or by "date" if that's more accurate

    # Separate home and away
    home_games = df[df["Location"] == "Home"].iloc[-22:].copy()
    away_games = df[df["Location"] == "Away"].iloc[-22:].copy()

    # Assign reverse numbering (1 = most recent)
    home_games["recent_num"] = range(len(home_games), 0, -1)
    away_games["recent_num"] = range(len(away_games), 0, -1)

    # Save
    last_games_by_team[team] = {
        "home": home_games[["game_id", "Location", "recent_num"]],
        "away": away_games[["game_id", "Location", "recent_num"]]
    }

last_games_by_team

{'ATL': {'home':                                  game_id Location  recent_num
  52  845014e2-2dec-425f-9e66-bd89bae7542e     Home          22
  73  86fe5811-d2e9-4488-be09-4d3cf89c0b46     Home          21
  44  89cb2451-25db-4ae4-95ef-3bcc44f328c8     Home          20
  71  8c1d1f48-9895-4c00-8c11-2d699e39e115     Home          19
  53  8cb21d2f-cded-4dc7-a788-88eb93ba1bb9     Home          18
  11  8f81ebea-2f4b-4298-9be6-74cca8414618     Home          17
  35  90175857-7de2-40f1-a01e-ece8d73fb2d3     Home          16
  39  a30d8477-78ec-4290-b1e8-f40ea7b99543     Home          15
  72  a7419964-379c-4d02-a789-87568949a4df     Home          14
  30  ae8cf5cb-5c5a-4694-b665-8c776cd1ae1d     Home          13
  8   b841ddbe-e331-4d47-95fd-d5fbcd18464c     Home          12
  69  b9120d08-4d53-4dba-abf2-9a12e34132ab     Home          11
  56  ba52d0bb-f224-4198-9e4f-c346446e85ed     Home          10
  10  c2a99b2c-c937-4bf7-8557-390b6cc6cb74     Home           9
  79  c48ce4ce-4b5a-4447-

### Player DF reduced with context games

In [15]:
players_df = pd.read_csv(
    f"final_data/season_players_dataset_{year}_reduced/season_players_dataset_with_context_{year}.csv",
    encoding='utf-8-sig'
)

# Columns without prefix
sample_th_cols = [c for c in players_df.columns if c.startswith('th_')]
cols_no_prefix = [c.replace("th_", "") for c in sample_th_cols]

# Output DataFrames
test_dataset_home = pd.DataFrame(columns=cols_no_prefix + ["team"])
test_dataset_away = pd.DataFrame(columns=cols_no_prefix + ["team"])

for team in teams:
    # Load games for this team and sort chronologically
    team_games = pd.read_csv(f"raw_data/season_team_stats_{year}/{team}.csv", encoding='utf-8')
    team_games = team_games.sort_values(by="game_id").reset_index(drop=True)

    # HOME games
    home_games = team_games[team_games["Location"] == "Home"].iloc[-22:]
    if len(home_games) == 22:
        # Games 1–10 (most recent) for main stats
        main_game_ids = home_games.iloc[-10:]["game_id"].tolist()
        avg_stats = players_df[players_df["game_id"].isin(main_game_ids)].mean(numeric_only=True)

        # Context games from older games (games 11–22)
        old_games = home_games.iloc[:-10]  # first 12 rows of these 22
        contexts = {}
        for i in range(4):
            ctx_game_ids = old_games.iloc[i*3:(i+1)*3]["game_id"].tolist()
            ctx_avg = players_df[players_df["game_id"].isin(ctx_game_ids)].mean(numeric_only=True)
            for col in cols_no_prefix:
                contexts[f"context{i+1}_{col}"] = ctx_avg[f"th_{col}"]

        row = {"team": team}
        for col in cols_no_prefix:
            row[col] = avg_stats[f"th_{col}"]
        row.update(contexts)
        test_dataset_home.loc[len(test_dataset_home)] = row

    # AWAY games
    away_games = team_games[team_games["Location"] == "Away"].iloc[-22:]
    if len(away_games) == 22:
        main_game_ids = away_games.iloc[-10:]["game_id"].tolist()
        avg_stats = players_df[players_df["game_id"].isin(main_game_ids)].mean(numeric_only=True)

        old_games = away_games.iloc[:-10]
        contexts = {}
        for i in range(4):
            ctx_game_ids = old_games.iloc[i*3:(i+1)*3]["game_id"].tolist()
            ctx_avg = players_df[players_df["game_id"].isin(ctx_game_ids)].mean(numeric_only=True)
            for col in cols_no_prefix:
                contexts[f"context{i+1}_{col}"] = ctx_avg[f"ta_{col}"]

        row = {"team": team}
        for col in cols_no_prefix:
            row[col] = avg_stats[f"ta_{col}"]
        row.update(contexts)
        test_dataset_away.loc[len(test_dataset_away)] = row

# Save results
output_dir = f"final_data/season_players_dataset_{year}_reduced/test_dataset"
os.makedirs(output_dir, exist_ok=True)
test_dataset_home.to_csv(f"{output_dir}/test_dataset_home_{year}.csv", index=False, encoding='utf-8-sig')
test_dataset_away.to_csv(f"{output_dir}/test_dataset_away_{year}.csv", index=False, encoding='utf-8-sig')

### Player DF with context games

In [16]:
players_df = pd.read_csv(
    f"final_data/season_players_dataset_{year}/season_players_dataset_with_context_{year}.csv",
    encoding='utf-8-sig'
)

# Columns without prefix
sample_th_cols = [c for c in players_df.columns if c.startswith('th_')]
cols_no_prefix = [c.replace("th_", "") for c in sample_th_cols]

# Output DataFrames
test_dataset_home = pd.DataFrame(columns=cols_no_prefix + ["team"])
test_dataset_away = pd.DataFrame(columns=cols_no_prefix + ["team"])

for team in teams:
    # Load games for this team and sort chronologically
    team_games = pd.read_csv(f"raw_data/season_team_stats_{year}/{team}.csv", encoding='utf-8')
    team_games = team_games.sort_values(by="game_id").reset_index(drop=True)

    # HOME games
    home_games = team_games[team_games["Location"] == "Home"].iloc[-22:]
    if len(home_games) == 22:
        # Games 1–10 (most recent) for main stats
        main_game_ids = home_games.iloc[-10:]["game_id"].tolist()
        avg_stats = players_df[players_df["game_id"].isin(main_game_ids)].mean(numeric_only=True)

        # Context games from older games (games 11–22)
        old_games = home_games.iloc[:-10]  # first 12 rows of these 22
        contexts = {}
        for i in range(4):
            ctx_game_ids = old_games.iloc[i*3:(i+1)*3]["game_id"].tolist()
            ctx_avg = players_df[players_df["game_id"].isin(ctx_game_ids)].mean(numeric_only=True)
            for col in cols_no_prefix:
                contexts[f"context{i+1}_{col}"] = ctx_avg[f"th_{col}"]

        row = {"team": team}
        for col in cols_no_prefix:
            row[col] = avg_stats[f"th_{col}"]
        row.update(contexts)
        test_dataset_home.loc[len(test_dataset_home)] = row

    # AWAY games
    away_games = team_games[team_games["Location"] == "Away"].iloc[-22:]
    if len(away_games) == 22:
        main_game_ids = away_games.iloc[-10:]["game_id"].tolist()
        avg_stats = players_df[players_df["game_id"].isin(main_game_ids)].mean(numeric_only=True)

        old_games = away_games.iloc[:-10]
        contexts = {}
        for i in range(4):
            ctx_game_ids = old_games.iloc[i*3:(i+1)*3]["game_id"].tolist()
            ctx_avg = players_df[players_df["game_id"].isin(ctx_game_ids)].mean(numeric_only=True)
            for col in cols_no_prefix:
                contexts[f"context{i+1}_{col}"] = ctx_avg[f"ta_{col}"]

        row = {"team": team}
        for col in cols_no_prefix:
            row[col] = avg_stats[f"ta_{col}"]
        row.update(contexts)
        test_dataset_away.loc[len(test_dataset_away)] = row

# Save results
output_dir = f"final_data/season_players_dataset_{year}/test_dataset"
os.makedirs(output_dir, exist_ok=True)
test_dataset_home.to_csv(f"{output_dir}/test_dataset_home_{year}.csv", index=False, encoding='utf-8-sig')
test_dataset_away.to_csv(f"{output_dir}/test_dataset_away_{year}.csv", index=False, encoding='utf-8-sig')

### Teams DF with context games

In [ ]:
teams_df = pd.read_csv(
    f"final_data/season_teams_dataset_{year}/season_teams_dataset_with_context_{year}.csv",
    encoding='utf-8-sig'
)

# Columns without prefix
sample_th_cols = [c for c in teams_df.columns if c.startswith('team_home_')]
cols_no_prefix = [c.replace("team_home_", "") for c in sample_th_cols]
cols_no_prefix = [c for c in cols_no_prefix if not c == "wins"] 

# Output DataFrames
test_dataset_home = pd.DataFrame(columns=cols_no_prefix + ["team"])
test_dataset_away = pd.DataFrame(columns=cols_no_prefix + ["team"])

for team in teams:
    # Load games for this team and sort chronologically
    team_games = pd.read_csv(f"raw_data/season_team_stats_{year}/{team}.csv", encoding='utf-8')
    team_games = team_games.sort_values(by="game_id").reset_index(drop=True)

    # HOME games
    home_games = team_games[team_games["Location"] == "Home"].iloc[-22:]
    if len(home_games) == 22:
        # Games 1–10 (most recent) for main stats
        main_game_ids = home_games.iloc[-10:]["game_id"].tolist()
        avg_stats = teams_df[teams_df["game_id"].isin(main_game_ids)].mean(numeric_only=True)

        # Context games from older games (games 11–22)
        old_games = home_games.iloc[:-10]  # first 12 rows of these 22
        contexts = {}
        for i in range(4):
            ctx_game_ids = old_games.iloc[i*3:(i+1)*3]["game_id"].tolist()
            ctx_avg = teams_df[teams_df["game_id"].isin(ctx_game_ids)].mean(numeric_only=True)
            for col in cols_no_prefix:
                contexts[f"context{i+1}_{col}"] = ctx_avg[f"team_home_{col}"]

        row = {"team": team}
        for col in cols_no_prefix:
            row[col] = avg_stats[f"team_home_{col}"]
        row.update(contexts)
        test_dataset_home.loc[len(test_dataset_home)] = row

    # AWAY games
    away_games = team_games[team_games["Location"] == "Away"].iloc[-22:]
    if len(away_games) == 22:
        main_game_ids = away_games.iloc[-10:]["game_id"].tolist()
        avg_stats = teams_df[teams_df["game_id"].isin(main_game_ids)].mean(numeric_only=True)

        old_games = away_games.iloc[:-10]
        contexts = {}
        for i in range(4):
            ctx_game_ids = old_games.iloc[i*3:(i+1)*3]["game_id"].tolist()
            ctx_avg = teams_df[teams_df["game_id"].isin(ctx_game_ids)].mean(numeric_only=True)
            for col in cols_no_prefix:
                contexts[f"context{i+1}_{col}"] = ctx_avg[f"team_away_{col}"]

        row = {"team": team}
        for col in cols_no_prefix:
            row[col] = avg_stats[f"team_away_{col}"]
        row.update(contexts)
        test_dataset_away.loc[len(test_dataset_away)] = row

# Save results
output_dir = f"final_data/season_teams_dataset_{year}/test_dataset"
os.makedirs(output_dir, exist_ok=True)
test_dataset_home.to_csv(f"{output_dir}/test_dataset_home_{year}.csv", index=False, encoding='utf-8-sig')
test_dataset_away.to_csv(f"{output_dir}/test_dataset_away_{year}.csv", index=False, encoding='utf-8-sig')

### Teams DF reduced with context games

In [20]:
teams_df = pd.read_csv(
    f"final_data/season_teams_dataset_{year}_reduced/season_teams_dataset_with_context_{year}.csv",
    encoding='utf-8-sig'
)

# Columns without prefix
sample_th_cols = [c for c in teams_df.columns if c.startswith('team_home_')]
cols_no_prefix = [c.replace("team_home_", "") for c in sample_th_cols]
cols_no_prefix = [c for c in cols_no_prefix if not c == "wins"]

# Output DataFrames
test_dataset_home = pd.DataFrame(columns=cols_no_prefix + ["team"])
test_dataset_away = pd.DataFrame(columns=cols_no_prefix + ["team"])

for team in teams:
    # Load games for this team and sort chronologically
    team_games = pd.read_csv(f"raw_data/season_team_stats_{year}/{team}.csv", encoding='utf-8')
    team_games = team_games.sort_values(by="game_id").reset_index(drop=True)

    # HOME games
    home_games = team_games[team_games["Location"] == "Home"].iloc[-22:]
    if len(home_games) == 22:
        # Games 1–10 (most recent) for main stats
        main_game_ids = home_games.iloc[-10:]["game_id"].tolist()
        avg_stats = teams_df[teams_df["game_id"].isin(main_game_ids)].mean(numeric_only=True)

        # Context games from older games (games 11–22)
        old_games = home_games.iloc[:-10]  # first 12 rows of these 22
        contexts = {}
        for i in range(4):
            ctx_game_ids = old_games.iloc[i*3:(i+1)*3]["game_id"].tolist()
            ctx_avg = teams_df[teams_df["game_id"].isin(ctx_game_ids)].mean(numeric_only=True)
            for col in cols_no_prefix:
                contexts[f"context{i+1}_{col}"] = ctx_avg[f"team_home_{col}"]

        row = {"team": team}
        for col in cols_no_prefix:
            row[col] = avg_stats[f"team_home_{col}"]
        row.update(contexts)
        test_dataset_home.loc[len(test_dataset_home)] = row

    # AWAY games
    away_games = team_games[team_games["Location"] == "Away"].iloc[-22:]
    if len(away_games) == 22:
        main_game_ids = away_games.iloc[-10:]["game_id"].tolist()
        avg_stats = teams_df[teams_df["game_id"].isin(main_game_ids)].mean(numeric_only=True)

        old_games = away_games.iloc[:-10]
        contexts = {}
        for i in range(4):
            ctx_game_ids = old_games.iloc[i*3:(i+1)*3]["game_id"].tolist()
            ctx_avg = teams_df[teams_df["game_id"].isin(ctx_game_ids)].mean(numeric_only=True)
            for col in cols_no_prefix:
                contexts[f"context{i+1}_{col}"] = ctx_avg[f"team_away_{col}"]

        row = {"team": team}
        for col in cols_no_prefix:
            row[col] = avg_stats[f"team_away_{col}"]
        row.update(contexts)
        test_dataset_away.loc[len(test_dataset_away)] = row

# Save results
output_dir = f"final_data/season_teams_dataset_{year}_reduced/test_dataset"
os.makedirs(output_dir, exist_ok=True)
test_dataset_home.to_csv(f"{output_dir}/test_dataset_home_{year}.csv", index=False, encoding='utf-8-sig')
test_dataset_away.to_csv(f"{output_dir}/test_dataset_away_{year}.csv", index=False, encoding='utf-8-sig')